# T10 — ARIMA Model — Book: CH07

**Methodology**: Marco Peixeiro, *Time Series Forecasting in Python*, Chapter 7.

### Book-mandated steps (CH07):
1. ADF at level + diff-1 + diff-2 → determine d
2. ACF + PACF on differenced series
3. `optimize_ARIMA(endog, order_list, d)` → select (p,q) by lowest AIC
4. Fit SARIMAX(p,d,q) → Ljung-Box + QQ plot (CH07 adds QQ)
5. `rolling_forecast_engine` → walk-forward validation
6. Full test evaluation

In [ ]:
import sys, os
from pathlib import Path
from functools import partial
from itertools import product

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Book imports — exactly as CH07 uses them
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.graphics.gofplots import qqplot          # CH07 addition
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller

from src.models.classical import (
    build_pca_health_index, compute_failure_threshold,
    run_stationarity_report, check_stationarity_adf,
    plot_acf_pacf, smooth_series,
    optimize_ARIMA, check_residuals,
    rolling_forecast_engine, predict_rul_arima,
    predict_dataset, RUL_CAP,
)
from src.evaluation.metrics import evaluate, evaluate_per_subset

PROC_DIR    = ROOT / "data" / "processed"
SENSOR_COLS = [f"s{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

## 1. Load data + build health_index

In [ ]:
train = pd.read_csv(PROC_DIR / "train_features.csv")
test  = pd.read_csv(PROC_DIR / "test_features.csv")

train, test = build_pca_health_index(train, test, SENSOR_COLS)
THRESHOLD_MAP = compute_failure_threshold(train, end_of_life_rul=5, quantile=0.5)
print("Thresholds:", THRESHOLD_MAP)

## 2. ADF at level + diff-1 + diff-2 → determine d (CH07 rule)

Book tests up to second difference if first is still non-stationary. d is NOT hardcoded.

In [ ]:
# CH07 pattern: test at level, diff-1, diff-2 for each sampled engine
# Modal recommended_d across all engines = the d we pass to optimize_ARIMA
stationarity_df = run_stationarity_report(train, n_engines_per_subset=5)

# Extract modal d from report
MODAL_D = 1
print(f"\nUsing d = {MODAL_D} for ARIMA (modal across all sampled engines)")

In [ ]:
# ── After running stationarity_df = run_stationarity_report(...) ──────

# 1. See d distribution across all engines
d_counts = stationarity_df["recommended_d"].value_counts().sort_index()
print("\nd=0 (already stationary)   :", d_counts.get(0, 0), "engines")
print("d=1 (1 difference needed)  :", d_counts.get(1, 0), "engines")
print("d=2 (2 differences needed) :", d_counts.get(2, 0), "engines")

# 2. Extract the modal d — this is what you pass to ARIMA
MODAL_D = int(stationarity_df["recommended_d"].mode()[0])  # = 2
print(f"Using d = {MODAL_D}")   # should print 2, not 1


# 3. Visual: compare raw vs differenced for one engine
eid  = train[train.dataset_id == 1].engine_id.iloc[0]
raw  = train[train.engine_id == eid].sort_values("cycle").health_index.values
smth = smooth_series(raw, window=5)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].plot(smth)
axes[0].set_title("health_index — RAW (d=0)")
axes[0].set_xlabel("cycle")

diff1 = np.diff(smth, n=1)
axes[1].plot(diff1, color="orange")
axes[1].axhline(0, color="red", ls="--")
axes[1].set_title("health_index — after diff-1 (d=1)")
axes[1].set_xlabel("cycle")

plt.tight_layout()
plt.show()
# Raw: should show a clear downward trend (non-stationary)
# After diff-1: should hover around 0 with no trend (stationary)

## 3. ADF demo on one engine (CH07 verbatim pattern)

In [ ]:
# CH07 verbatim: show ADF at level, diff-1, diff-2 for one engine
eid  = train[train.dataset_id == 1].engine_id.iloc[0]
raw  = train[train.engine_id == eid].sort_values("cycle").health_index.values
smth = smooth_series(raw, window=5)

result = check_stationarity_adf(smth)
print(f"Level  ADF p-value : {result['level_pvalue']}")
print(f"Diff-1 ADF p-value : {result['diff1_pvalue']}")
print(f"Diff-2 ADF p-value : {result['diff2_pvalue']}")
print(f"Recommended d      : {result['recommended_d']}")

## 4. ACF + PACF on differenced series (CH07)

After applying d differences, ACF/PACF guides selection of p and q.

In [ ]:
# After differencing d times, inspect ACF + PACF to get (p, q) candidates
if MODAL_D > 0:
    series_for_acf = np.diff(smth, n=MODAL_D)
    plot_acf_pacf(series_for_acf, lags=20,
                  title=f"health_index (diff-{MODAL_D}) — engine {eid}")
else:
    plot_acf_pacf(smth, lags=20, title=f"health_index — engine {eid}")

## 5. `optimize_ARIMA` — select (p,q) by AIC (CH07 core step)

Exact copy of CH07 `optimize_ARIMA` function. d is fixed from ADF above.

In [ ]:
# CH07 exactly: optimize_ARIMA fixes d from ADF, grid-searches p and q by AIC
order_list = list(product(range(1, 4), range(1, 4)))  # (p,q) in [1,2,3] x [1,2,3]

arima_aic_df = optimize_ARIMA(smth, order_list, d=MODAL_D)
print(arima_aic_df.to_string(index=False))

best_pq = arima_aic_df.iloc[0]["(p,q)"]
BEST_P, BEST_Q = int(best_pq[0]), int(best_pq[1])
print(f"\nBest ARIMA order: ({BEST_P},{MODAL_D},{BEST_Q}) — lowest AIC")

## 6. Fit best ARIMA + Ljung-Box + QQ plot (CH07 requirement)

CH07 adds QQ plot on top of Ljung-Box (CH06 had only Ljung-Box).

In [ ]:
# CH07: fit, print summary, run Ljung-Box AND QQ plot
model_fit = SARIMAX(smth, order=(BEST_P, MODAL_D, BEST_Q), simple_differencing=False).fit(disp=False)
print(model_fit.summary())

# CH07 adds QQ plot in addition to Ljung-Box (plot_qq=True)
lb_result = check_residuals(
    model_fit.resid,
    model_name=f"ARIMA({BEST_P},{MODAL_D},{BEST_Q})",
    plot_qq=True,   # CH07 requirement — not in CH05/CH06
)

## 7. Forecast trajectory demo (CH07 style)

In [ ]:
# CH07 demo: show the actual forecast trajectory on one engine
model_demo  = SARIMAX(smth, order=(BEST_P, MODAL_D, BEST_Q), simple_differencing=False).fit(disp=False)
fcst        = model_demo.forecast(steps=100)
thresh_demo = THRESHOLD_MAP.get(1, 0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(smth)), smth, color="lightgray", label="raw smoothed")
ax.plot(range(len(smth), len(smth) + len(fcst)), fcst,
        color="darkorange", label=f"ARIMA({BEST_P},{MODAL_D},{BEST_Q}) forecast")
ax.axhline(thresh_demo, color="red", ls="--", label=f"threshold={thresh_demo:.2f}")
ax.set_xlabel("cycle"); ax.set_ylabel("health_index")
ax.set_title(f"ARIMA forecast — engine {eid}")
ax.legend(); plt.tight_layout(); plt.show()

## 8. Rolling forecast — walk-forward (CH07 pattern)

In [ ]:
TRAIN_LEN = int(len(smth) * 0.7)
WINDOW    = 1

pred_arima = rolling_forecast_engine(
    series    = smth,
    train_len = TRAIN_LEN,
    order     = (BEST_P, MODAL_D, BEST_Q),
    window    = WINDOW,
)
actual_val = smth[TRAIN_LEN: TRAIN_LEN + len(pred_arima)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(smth, color="steelblue", label="observed health_index")
ax.plot(range(TRAIN_LEN, TRAIN_LEN + len(pred_arima)), pred_arima,
        color="darkorange", ls="--",
        label=f"ARIMA({BEST_P},{MODAL_D},{BEST_Q}) rolling forecast")
ax.axvline(TRAIN_LEN, color="gray", ls=":", label="train/val split")
ax.axhline(THRESHOLD_MAP.get(1,0), color="red", ls="--", label="threshold")
ax.set_xlabel("cycle"); ax.set_ylabel("health_index")
ax.legend(); plt.tight_layout(); plt.show()

rmse_roll = float(np.sqrt(np.mean((actual_val - pred_arima)**2)))
print(f"Rolling forecast RMSE on health_index: {rmse_roll:.4f}")

## 9. Full test-set evaluation

In [ ]:
predict_fn = partial(predict_rul_arima, p=BEST_P, d=MODAL_D, q=BEST_Q)
y_true, y_pred, dids = predict_dataset(test, predict_fn, THRESHOLD_MAP)

print(f"=== ARIMA({BEST_P},{MODAL_D},{BEST_Q}) Test Results ===")
test_metrics = evaluate_per_subset(y_true, y_pred, dids, model_name=f"ARIMA({BEST_P},{MODAL_D},{BEST_Q})")
display(test_metrics)